# Поиск характерных терминов в публикациях по квантовой физике: TF-IDF и KeyBERT

В работе анализируются публикации категории **Quantum Physics (`quant-ph`)** на arXiv.

Для сравнения сформированы две выборки: 50 статей за июль 2026 года и 50 статей за предшествующие два года. Цель анализа - выделить слова и словосочетания, характерные для новых публикаций, двумя независимыми методами и определить их пересечение.

Этапы анализа:
1. формирование корпуса публикаций;
2. получение полных текстов;
3. предобработка текстов;
4. извлечение терминов с помощью TF-IDF;
5. извлечение ключевых слов и словосочетаний с помощью KeyBERT;
6. сопоставление результатов двух методов.


## 1. Импорт библиотек и настройка параметров

Подключаем библиотеки для загрузки и обработки данных, извлечения текста из HTML и PDF, анализа TF-IDF и работы с KeyBERT. В начале также задаются основные параметры анализа и фиксируется `random seed` для воспроизводимости случайной выборки.


In [1]:
%pip install -q pandas requests beautifulsoup4 pypdf scikit-learn keybert sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from io import BytesIO
import random
import re
import time

import pandas as pd
import requests

from bs4 import BeautifulSoup
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT

DATA_DIR = Path(".")

RANDOM_SEED = 42
N_PER_GROUP = 50
TFIDF_THRESHOLD = 0.6
KEYBERT_THRESHOLD = 0.4

random.seed(RANDOM_SEED)

print("Рабочая папка:", DATA_DIR.resolve())


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Рабочая папка: C:\Users\user\Desktop\quantum_physics


## 2. Получение метаданных с arXiv

Источником данных служат архивные страницы arXiv категории `quant-ph`. Для каждого месяца извлекаются идентификатор arXiv, название публикации и предметные категории.


In [3]:
def fetch_month_from_arxiv(year, month, retries=4):
    month_label = f"{year}-{month:02d}"
    url = f"https://arxiv.org/list/quant-ph/{month_label}?show=2000"
    headers = {"User-Agent": "Mozilla/5.0 (academic research; quant-ph topic study)"}

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=120)
            print(f"{month_label}: HTTP {r.status_code}")
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, 'html.parser')
                rows = []
                for dt, dd in zip(soup.find_all('dt'), soup.find_all('dd')):
                    abs_link = dt.find('a', href=re.compile(r'^/abs/'))
                    if abs_link is None:
                        continue
                    arxiv_id = re.sub(r'v\d+$', '', abs_link['href'].replace('/abs/', ''))
                    title_div = dd.find('div', class_='list-title')
                    subjects_div = dd.find('div', class_='list-subjects')
                    title = title_div.get_text(' ', strip=True) if title_div else ''
                    subjects = subjects_div.get_text(' ', strip=True) if subjects_div else ''
                    title = re.sub(r'^Title:\s*', '', title)
                    subjects = re.sub(r'^Subjects:\s*', '', subjects)
                    rows.append({'arxiv_id': arxiv_id, 'title': title,
                                 'subjects': subjects, 'month': month_label})

                rows = [x for x in rows if 'Quantum Physics (quant-ph)' in x['subjects']]
                rows = list({x['arxiv_id']: x for x in rows}.values())
                if rows:
                    print(f"{month_label}: найдено {len(rows)} quant-ph статей")
                    return rows
        except requests.RequestException as exc:
            print('Ошибка сети:', exc)
        if attempt < retries:
            time.sleep(5 * attempt)
    raise RuntimeError(f"Не удалось получить arXiv за {month_label}")


### Проверка получения данных

Перед формированием корпуса проверяем получение списка публикаций за июль 2026 года. Успешный ответ сервера и непустой список подтверждают, что данные доступны для дальнейшей обработки.


In [4]:
test_rows = fetch_month_from_arxiv(2026, 7)
print('Статей quant-ph за июль 2026:', len(test_rows))
display(pd.DataFrame(test_rows).head())


2026-07: HTTP 200
2026-07: найдено 1930 quant-ph статей
Статей quant-ph за июль 2026: 1930


,arxiv_id,title,subjects,month
0,2607.00040,Quantum Amplitude Estimation in Gradient-Based...,Quantum Physics (quant-ph),2026-07
1,2607.00045,Quantum Reconstruction and Phenomenology per t...,Quantum Physics (quant-ph),2026-07
2,2607.00059,"Comment on ""Ideal clocks -- a convenient ficti...",Quantum Physics (quant-ph) ; General Relativit...,2026-07
3,2607.00063,Spectral Geometry and Bosonic-Bloch Probes: Ex...,Quantum Physics (quant-ph) ; Artificial Intell...,2026-07
4,2607.00100,Velocity of a Quantum Particle in a Classicall...,Quantum Physics (quant-ph),2026-07


## 3. Формирование корпуса

Формируются две группы по 50 публикаций:

- **JULY** - случайная выборка из публикаций за июль 2026 года;
- **PREVIOUS** - публикации за период с июля 2024 по июнь 2026 года.

Для группы `PREVIOUS` выбираются по две публикации из каждого месяца и по одной дополнительной публикации из первого и последнего месяца. Это позволяет распределить 50 текстов по всему двухлетнему периоду, а не концентрировать их в отдельных месяцах.

Фиксированный `random seed` обеспечивает воспроизводимость выборки при повторном запуске.


In [5]:
# Период для сравнительной выборки: июль 2024 - июнь 2026
months = pd.period_range("2024-07", "2026-06", freq="M")

previous_rows = []

for month_number, month in enumerate(months):
    # Обычно берём по 2 статьи в месяц.
    articles_to_take = 2

    # Чтобы за 24 месяца получилось ровно 50 статей,
    # в первом и последнем месяце берём по 3 статьи.
    if month_number == 0 or month_number == len(months) - 1:
        articles_to_take = 3

    candidates = fetch_month_from_arxiv(month.year, month.month)
    selected_articles = random.sample(candidates, articles_to_take)

    for article in selected_articles:
        article["period"] = "PREVIOUS"
        previous_rows.append(article)

    print(
        f"PREVIOUS {month}: +{articles_to_take}, "
        f"всего {len(previous_rows)}/50"
    )

    time.sleep(2)


# Отдельно выбираем 50 статей за июль 2026
july_candidates = fetch_month_from_arxiv(2026, 7)
july_rows = random.sample(july_candidates, N_PER_GROUP)

for article in july_rows:
    article["period"] = "JULY"


# Объединяем обе группы в одну таблицу
all_rows = previous_rows + july_rows
corpus_df = pd.DataFrame(all_rows)

print("\nРазмер корпуса:", len(corpus_df))
print(corpus_df["period"].value_counts())


# Простые проверки перед дальнейшей работой
if len(corpus_df) != 100:
    raise ValueError("В корпусе должно быть 100 статей")

if corpus_df["arxiv_id"].nunique() != 100:
    raise ValueError("В корпусе обнаружены повторяющиеся arXiv ID")


# Сохраняем список выбранных публикаций
metadata_file = DATA_DIR / "physics_corpus_metadata.csv"
corpus_df.to_csv(metadata_file, index=False, encoding="utf-8-sig")

display(corpus_df.head())


2024-07: HTTP 200
2024-07: найдено 1205 quant-ph статей
PREVIOUS 2024-07: +3, всего 3/50
2024-08: HTTP 200
2024-08: найдено 1102 quant-ph статей
PREVIOUS 2024-08: +2, всего 5/50
2024-09: HTTP 200
2024-09: найдено 1110 quant-ph статей
PREVIOUS 2024-09: +2, всего 7/50
2024-10: HTTP 200
2024-10: найдено 1303 quant-ph статей
PREVIOUS 2024-10: +2, всего 9/50
2024-11: HTTP 200
2024-11: найдено 1199 quant-ph статей
PREVIOUS 2024-11: +2, всего 11/50
2024-12: HTTP 200
2024-12: найдено 1247 quant-ph статей
PREVIOUS 2024-12: +2, всего 13/50
2025-01: HTTP 200
2025-01: найдено 1091 quant-ph статей
PREVIOUS 2025-01: +2, всего 15/50
2025-02: HTTP 200
2025-02: найдено 1071 quant-ph статей
PREVIOUS 2025-02: +2, всего 17/50
2025-03: HTTP 200
2025-03: найдено 1318 quant-ph статей
PREVIOUS 2025-03: +2, всего 19/50
2025-04: HTTP 200
2025-04: найдено 1319 quant-ph статей
PREVIOUS 2025-04: +2, всего 21/50
2025-05: HTTP 200
2025-05: найдено 1236 quant-ph статей
PREVIOUS 2025-05: +2, всего 23/50
2025-06: HTTP 

,arxiv_id,title,subjects,month,period
0,2407.06006,Heisenberg-limited Bayesian phase estimation w...,Quantum Physics (quant-ph),2024-07,PREVIOUS
1,2407.01522,A diagrammatic language for the Causaloid fram...,Quantum Physics (quant-ph) ; General Relativit...,2024-07,PREVIOUS
2,2407.14195,Optimized surface ion trap design for tight co...,Quantum Physics (quant-ph) ; Atomic Physics (p...,2024-07,PREVIOUS
3,2408.10801,Solving an Industrially Relevant Quantum Chemi...,Quantum Physics (quant-ph),2024-08,PREVIOUS
4,2408.09922,Realization of Landau-Zener Rabi Oscillations ...,Quantum Physics (quant-ph) ; Atomic Physics (p...,2024-08,PREVIOUS


## 4. Получение полных текстов

Для каждой публикации сначала запрашивается HTML-версия arXiv. Если она недоступна, текст извлекается из PDF.

Полный текст и способ его получения сохраняются в таблице. Промежуточное сохранение корпуса выполняется через каждые пять публикаций, чтобы уже загруженные данные не потерялись при возможном сетевом сбое.


In [6]:
def get_text_from_html(arxiv_id):
    url = f"https://arxiv.org/html/{arxiv_id}"
    headers = {"User-Agent": "Mozilla/5.0 academic-research"}

    response = requests.get(url, headers=headers, timeout=90)

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Удаляем элементы страницы, которые не относятся к тексту статьи.
    unwanted_tags = soup(["script", "style", "nav"])

    for tag in unwanted_tags:
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    if len(text) < 1000:
        return None

    return text


def get_text_from_pdf(arxiv_id):
    url = f"https://arxiv.org/pdf/{arxiv_id}"
    headers = {"User-Agent": "Mozilla/5.0 academic-research"}

    response = requests.get(url, headers=headers, timeout=120)

    if response.status_code != 200:
        return None

    pdf_file = BytesIO(response.content)
    reader = PdfReader(pdf_file)

    pages_text = []

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text is not None:
            pages_text.append(page_text)

    text = "\n".join(pages_text)

    if len(text) < 1000:
        return None

    return text


In [7]:
corpus_df["full_text"] = None
corpus_df["text_source"] = None

checkpoint_file = DATA_DIR / "physics_corpus_checkpoint.pkl"

for position, row_index in enumerate(corpus_df.index, start=1):
    arxiv_id = corpus_df.at[row_index, "arxiv_id"]

    text = None
    source = None

    # Сначала пробуем HTML.
    try:
        text = get_text_from_html(arxiv_id)

        if text is not None:
            source = "html"

    except Exception as error:
        print("Ошибка HTML:", arxiv_id, error)

    # Если HTML получить не удалось, пробуем PDF.
    if text is None:
        try:
            text = get_text_from_pdf(arxiv_id)

            if text is not None:
                source = "pdf"

        except Exception as error:
            print("Ошибка PDF:", arxiv_id, error)

    corpus_df.at[row_index, "full_text"] = text
    corpus_df.at[row_index, "text_source"] = source

    if text is None:
        text_length = 0
    else:
        text_length = len(text)

    print(
        f"{position}/100 | {arxiv_id} | "
        f"{source} | {text_length:,} символов"
    )

    # Каждые 5 статей сохраняем промежуточный результат.
    if position % 5 == 0:
        corpus_df.to_pickle(checkpoint_file)

    time.sleep(2)


corpus_df.to_pickle(DATA_DIR / "physics_corpus_full.pkl")


1/100 | 2407.06006 | html | 209,522 символов
2/100 | 2407.01522 | html | 384,417 символов
3/100 | 2407.14195 | html | 33,316 символов
4/100 | 2408.10801 | html | 66,082 символов
5/100 | 2408.09922 | html | 32,637 символов
6/100 | 2409.06326 | html | 75,243 символов
7/100 | 2409.04863 | html | 62,299 символов
8/100 | 2410.11257 | html | 75,234 символов
9/100 | 2410.04169 | html | 72,168 символов
10/100 | 2411.19684 | html | 54,195 символов
11/100 | 2411.02166 | html | 72,785 символов
12/100 | 2412.01851 | html | 115,626 символов
13/100 | 2412.04701 | html | 39,257 символов
14/100 | 2501.12018 | html | 36,445 символов
15/100 | 2501.12629 | html | 124,095 символов
16/100 | 2502.18236 | html | 74,332 символов
17/100 | 2502.01767 | html | 82,933 символов


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong 

18/100 | 2503.12580 | pdf | 68,558 символов
19/100 | 2503.10607 | html | 140,862 символов
20/100 | 2504.09234 | pdf | 48,379 символов
21/100 | 2504.19239 | html | 63,411 символов
22/100 | 2505.09714 | html | 67,569 символов
23/100 | 2505.24052 | html | 89,936 символов
24/100 | 2506.12906 | html | 150,090 символов
25/100 | 2506.14526 | html | 140,091 символов
26/100 | 2507.00504 | html | 55,777 символов
27/100 | 2507.08351 | html | 58,233 символов
28/100 | 2508.21486 | html | 357,335 символов
29/100 | 2508.17170 | html | 51,097 символов
30/100 | 2509.12670 | html | 76,963 символов
31/100 | 2509.07206 | html | 35,915 символов
32/100 | 2510.08299 | html | 73,969 символов
33/100 | 2510.24939 | html | 72,221 символов
34/100 | 2511.17047 | html | 60,160 символов
35/100 | 2511.05282 | html | 110,534 символов
36/100 | 2512.04494 | html | 52,728 символов
37/100 | 2512.16673 | html | 95,231 символов
38/100 | 2601.05113 | html | 55,971 символов
39/100 | 2601.18347 | html | 88,941 символов
40/100 

## 5. Проверка и предобработка текстов

Сначала проверяем наличие и длину полученных текстов. Затем создаём отдельное поле `analysis_text` для анализа: текст переводится в нижний регистр, удаляются некоторые технические элементы LaTeX, числа и лишние пробелы.

Исходный полный текст сохраняется без изменений.


In [8]:
corpus_df['text_length'] = corpus_df['full_text'].fillna('').str.len()
display(corpus_df.groupby('period')['text_length'].agg(['count','median','min','max']))
print(corpus_df['text_source'].value_counts(dropna=False))
assert corpus_df['full_text'].notna().all()
assert (corpus_df['text_length'] >= 1000).all()
print('Контроль качества пройден.')


,count,median,min,max
period,,,,
JULY,50,91747.5,14355,233751
PREVIOUS,50,72503.0,15776,384417


text_source
html    98
pdf      2
Name: count, dtype: int64
Контроль качества пройден.


In [9]:
LATEX_ARTIFACTS = [
    "displaystyle",
    "textstyle",
    "scriptstyle",
    "scriptscriptstyle",
    "begin",
    "end",
]


def clean_text(text):
    if text is None:
        return ""

    # Приводим весь текст к нижнему регистру.
    cleaned = text.lower()

    # Удаляем несколько технических слов LaTeX.
    for token in LATEX_ARTIFACTS:
        pattern = rf"\b{re.escape(token)}\b"
        cleaned = re.sub(pattern, " ", cleaned)

    # Удаляем команды LaTeX вида \alpha, \text и т. п.
    cleaned = re.sub(r"\\[a-zA-Z]+", " ", cleaned)

    # Удаляем отдельно стоящие числа.
    cleaned = re.sub(r"\b\d+\b", " ", cleaned)

    # Заменяем несколько пробелов одним.
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned.strip()


analysis_texts = []

for text in corpus_df["full_text"]:
    cleaned_text = clean_text(text)
    analysis_texts.append(cleaned_text)

corpus_df["analysis_text"] = analysis_texts

print("Тексты подготовлены:", len(corpus_df))


Тексты подготовлены: 100


## 6. Извлечение терминов с помощью TF-IDF

Каждую из 50 июльских статей сравниваем с 50 публикациями за предыдущие два года.

Для этого к 50 предыдущим текстам по очереди добавляется одна июльская статья и строится `TfidfVectorizer`. Параметр `ngram_range=(1, 3)` означает, что рассматриваются отдельные слова, пары слов и сочетания из трёх слов.

Из текущей июльской статьи сохраняются слова и словосочетания с **TF-IDF score > 0.6**.


In [10]:
# Сначала отдельно получаем тексты двух групп.
previous_df = corpus_df[corpus_df["period"] == "PREVIOUS"]
july_df = corpus_df[corpus_df["period"] == "JULY"].copy()

previous_texts = previous_df["analysis_text"].tolist()

tfidf_rows = []


for article_number, (_, article) in enumerate(july_df.iterrows(), start=1):
    # Для одной июльской статьи создаём маленький корпус:
    # 50 предыдущих статей + текущая июльская статья.
    documents = previous_texts.copy()
    documents.append(article["analysis_text"])

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.95,
    )

    tfidf_matrix = vectorizer.fit_transform(documents)
    terms = vectorizer.get_feature_names_out()

    # Текущая июльская статья была добавлена последней,
    # поэтому её TF-IDF значения находятся в последней строке.
    july_scores = tfidf_matrix[-1].toarray()[0]

    found_in_article = 0

    # Проходим по всем терминам и их значениям обычным циклом.
    for term_index in range(len(terms)):
        term = terms[term_index]
        score = july_scores[term_index]

        if score > TFIDF_THRESHOLD:
            result = {
                "arxiv_id": article["arxiv_id"],
                "title": article["title"],
                "term": term,
                "tfidf_score": float(score),
            }

            tfidf_rows.append(result)
            found_in_article += 1

    print(
        f"TF-IDF {article_number}/50 | "
        f"найдено > {TFIDF_THRESHOLD}: {found_in_article}"
    )


tfidf_selected = pd.DataFrame(
    tfidf_rows,
    columns=["arxiv_id", "title", "term", "tfidf_score"],
)

tfidf_selected.to_csv(
    DATA_DIR / "tfidf_selected_over_06.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nВсего найденных строк TF-IDF:", len(tfidf_selected))

if len(tfidf_selected) > 0:
    print("Уникальных терминов:", tfidf_selected["term"].nunique())
    display(
        tfidf_selected
        .sort_values("tfidf_score", ascending=False)
        .head(20)
    )
else:
    print("При выбранном пороге TF-IDF термины не найдены.")


TF-IDF 1/50 | найдено > 0.6: 0
TF-IDF 2/50 | найдено > 0.6: 0
TF-IDF 3/50 | найдено > 0.6: 0
TF-IDF 4/50 | найдено > 0.6: 0
TF-IDF 5/50 | найдено > 0.6: 0
TF-IDF 6/50 | найдено > 0.6: 0
TF-IDF 7/50 | найдено > 0.6: 0
TF-IDF 8/50 | найдено > 0.6: 0
TF-IDF 9/50 | найдено > 0.6: 1
TF-IDF 10/50 | найдено > 0.6: 0
TF-IDF 11/50 | найдено > 0.6: 0
TF-IDF 12/50 | найдено > 0.6: 0
TF-IDF 13/50 | найдено > 0.6: 0
TF-IDF 14/50 | найдено > 0.6: 0
TF-IDF 15/50 | найдено > 0.6: 0
TF-IDF 16/50 | найдено > 0.6: 0
TF-IDF 17/50 | найдено > 0.6: 0
TF-IDF 18/50 | найдено > 0.6: 0
TF-IDF 19/50 | найдено > 0.6: 0
TF-IDF 20/50 | найдено > 0.6: 0
TF-IDF 21/50 | найдено > 0.6: 0
TF-IDF 22/50 | найдено > 0.6: 0
TF-IDF 23/50 | найдено > 0.6: 0
TF-IDF 24/50 | найдено > 0.6: 0
TF-IDF 25/50 | найдено > 0.6: 0
TF-IDF 26/50 | найдено > 0.6: 0
TF-IDF 27/50 | найдено > 0.6: 0
TF-IDF 28/50 | найдено > 0.6: 0
TF-IDF 29/50 | найдено > 0.6: 0
TF-IDF 30/50 | найдено > 0.6: 0
TF-IDF 31/50 | найдено > 0.6: 0
TF-IDF 32/50 | на

,arxiv_id,title,term,tfidf_score
0,2607.23745,The Physics of Unresolved Uncertainty: Quantum...,potentiality,0.707977
1,2607.06967,Tomography of a Macroscopic Quantum State infl...,sn,0.613910


## 7. Извлечение ключевых слов с помощью KeyBERT

KeyBERT применяется независимо от TF-IDF к каждой из 50 публикаций за июль 2026 года.

Для каждой статьи используется подготовленный полный текст. Параметр `keyphrase_ngram_range=(1, 3)` позволяет искать отдельные слова и словосочетания длиной до трёх слов. Для дальнейшего анализа сохраняются результаты с **KeyBERT score > 0.4**.

Модель создаётся один раз, после чего последовательно применяется к каждой статье.


### 7.1. Проверка количества кандидатов

Перед обработкой всех публикаций проверим, насколько параметр `top_n` влияет на применение заданного порога KeyBERT score > 0.4.

На одной тестовой статье рассматриваем 500 кандидатов и сравниваем минимальный score с выбранным порогом. Если минимальный score оказывается ниже 0.4, это означает, что в данном случае список кандидатов проходит через интересующую нас границу score.

Дополнительно минимальный score контролируется при обработке остальных статей, поскольку количество подходящих кандидатов может различаться между публикациями.

In [11]:
# Загружаем модель KeyBERT один раз
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

test_article = july_df.iloc[0]

test_keywords = kw_model.extract_keywords(
    test_article["analysis_text"],
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    top_n=500,
)

print("Всего получено кандидатов:", len(test_keywords))
print()

print("Первые 10 кандидатов:")
for phrase, score in test_keywords[:10]:
    print(phrase, round(score, 3))

print()

print("Последние 10 кандидатов:")
for phrase, score in test_keywords[-10:]:
    print(phrase, round(score, 3))

print()

lowest_score = test_keywords[-1][1]

above_04 = 0

for phrase, score in test_keywords:
    if score > KEYBERT_THRESHOLD:
        above_04 += 1

print("Минимальный score:", round(lowest_score, 3))
print(
    f"Кандидатов со score > {KEYBERT_THRESHOLD}:",
    above_04
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7351.92it/s]


Всего получено кандидатов: 500

Первые 10 кандидатов:
biharmonic equation discretization 0.705
discretizations biharmonic equation 0.703
biharmonic equation quantum 0.691
quantum biharmonic solver 0.686
biharmonic discretizations particular 0.683
biharmonic boundary value 0.676
discretization biharmonic 0.671
biharmonic problem qsvt 0.669
supported biharmonic equation 0.668
encodings discretizations biharmonic 0.666

Последние 10 кандидатов:
collect boundary values 0.362
symmetric dirichlet laplacian 0.361
boundary problem 0.361
quantum preconditioning 0.361
discretization error 0.361
boundary operators 0.361
boundary conditions periodic 0.361
contribution boundary values 0.361
prescribed consistent boundary 0.36
classical spectral discretization 0.36

Минимальный score: 0.36
Кандидатов со score > 0.4: 322


В тестовой статье среди 500 кандидатов минимальный score оказался ниже порога 0.4, поэтому для этой статьи выбранный размер списка охватывает границу интересующего нас порога

Для основной выборки сохраняется top_n=500 как практическое ограничение числа анализируемых кандидатов. При этом для некоторых публикаций все 500 полученных кандидатов могут иметь score выше 0.4. Следовательно, top_n=500 следует рассматривать как ограничение текущей реализации, а не как гарантию получения абсолютно всех возможных кандидатов выше порога

In [12]:
keybert_rows = []


# Последовательно анализируем каждую из 50 июльских статей
for article_number, (_, article) in enumerate(july_df.iterrows(), start=1):

    article_text = article["analysis_text"]

    # Получаем до 500 наиболее подходящих слов и словосочетаний.
    # Большое значение top_n нужно, чтобы результат не ограничивался
    # слишком маленьким числом кандидатов до применения порога 0.4.
    keywords = kw_model.extract_keywords(
        article_text,
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=500,
    )

    found_in_article = 0


    # KeyBERT возвращает пары:
    # (ключевое слово или словосочетание, score)
    for phrase, score in keywords:

        # Оставляем только результаты со score выше 0.4
        if score > KEYBERT_THRESHOLD:

            result = {
                "arxiv_id": article["arxiv_id"],
                "title": article["title"],
                "keyword": phrase,
                "keybert_score": float(score),
            }

            keybert_rows.append(result)
            found_in_article += 1


    # Проверяем минимальный score среди полученных кандидатов.
    # Если он ниже 0.4, значит список кандидатов дошёл
    # ниже интересующего нас порога.
    lowest_score = keywords[-1][1]

    print(
        f"KeyBERT {article_number}/50 | "
        f"найдено > {KEYBERT_THRESHOLD}: {found_in_article} | "
        f"минимальный score: {lowest_score:.3f}"
    )


# Собираем результаты всех 50 статей в одну таблицу
keybert_selected = pd.DataFrame(
    keybert_rows,
    columns=[
        "arxiv_id",
        "title",
        "keyword",
        "keybert_score",
    ],
)


# Сохраняем результаты
keybert_selected.to_csv(
    DATA_DIR / "keybert_july_over_04.csv",
    index=False,
    encoding="utf-8-sig",
)


# Выводим общую информацию
print()
print("Всего найденных строк KeyBERT:", len(keybert_selected))

if len(keybert_selected) > 0:

    unique_keywords = keybert_selected["keyword"].nunique()

    print(
        "Уникальных ключевых слов и словосочетаний:",
        unique_keywords
    )

    # Показываем 20 результатов с наибольшим score
    top_keywords = keybert_selected.sort_values(
        "keybert_score",
        ascending=False,
    ).head(20)

    display(top_keywords)

else:
    print(
        "При выбранном пороге KeyBERT "
        "ключевые слова и словосочетания не найдены."
    )

KeyBERT 1/50 | найдено > 0.4: 322 | минимальный score: 0.360
KeyBERT 2/50 | найдено > 0.4: 500 | минимальный score: 0.412
KeyBERT 3/50 | найдено > 0.4: 362 | минимальный score: 0.365
KeyBERT 4/50 | найдено > 0.4: 125 | минимальный score: 0.268
KeyBERT 5/50 | найдено > 0.4: 95 | минимальный score: 0.312
KeyBERT 6/50 | найдено > 0.4: 500 | минимальный score: 0.428
KeyBERT 7/50 | найдено > 0.4: 445 | минимальный score: 0.385
KeyBERT 8/50 | найдено > 0.4: 371 | минимальный score: 0.384
KeyBERT 9/50 | найдено > 0.4: 500 | минимальный score: 0.433
KeyBERT 10/50 | найдено > 0.4: 156 | минимальный score: 0.211
KeyBERT 11/50 | найдено > 0.4: 208 | минимальный score: 0.265
KeyBERT 12/50 | найдено > 0.4: 280 | минимальный score: 0.353
KeyBERT 13/50 | найдено > 0.4: 99 | минимальный score: 0.297
KeyBERT 14/50 | найдено > 0.4: 186 | минимальный score: 0.323
KeyBERT 15/50 | найдено > 0.4: 500 | минимальный score: 0.418
KeyBERT 16/50 | найдено > 0.4: 500 | минимальный score: 0.418
KeyBERT 17/50 | най

,arxiv_id,title,keyword,keybert_score
3220,2607.07970,Associating Trajectories with Quantum Processe...,trajectories quantum,0.7611
6822,2607.15201,Entanglement Detection for Two-Qubit and Three...,entanglement detection qubit,0.7540
14546,2607.15184,Backpropagating Pauli Propagation,backpropagating pauli propagation,0.7518
3221,2607.07970,Associating Trajectories with Quantum Processe...,associating trajectories quantum,0.7508
3963,2607.26978,Mean-field Pulse Adaptation for the Circulariz...,circularization interacting rydberg,0.7497
6823,2607.15201,Entanglement Detection for Two-Qubit and Three...,qubit entanglement detection,0.7433
12013,2607.24851,Unitary designs from perturbed time evolutions...,random unitary design,0.7380
10088,2607.27171,OQRAM: Oblivious Quantum Random Access Memory ...,secure quantum query,0.7353
3222,2607.07970,Associating Trajectories with Quantum Processe...,trajectories quantum processes,0.7351
1904,2607.17964,Scanless quantum Fourier-transform mid-infrare...,ir spectroscopy ftir,0.7327


## 8. Сопоставление результатов TF-IDF и KeyBERT

После независимого применения TF-IDF и KeyBERT сравним полученные результаты.

Для сопоставления приводим термины к нижнему регистру и удаляем лишние пробелы. Затем для каждой июльской статьи проверяем, какие слова и словосочетания были найдены одновременно двумя методами.

Сначала используем исходные пороги:
- TF-IDF score > 0.6
- KeyBERT score > 0.4

В результате считаем количество уникальных совпавших терминов и количество статей, в которых было найдено хотя бы одно совпадение.


In [13]:
def normalize_term(term):
    term = str(term).lower()
    term = re.sub(r"\s+", " ", term)
    return term.strip()


tfidf_for_merge = tfidf_selected.copy()
keybert_for_merge = keybert_selected.copy()


# Создаём одинаково названный столбец с нормализованными терминами.
tfidf_for_merge["normalized_term"] = tfidf_for_merge["term"].apply(normalize_term)
keybert_for_merge["normalized_term"] = keybert_for_merge["keyword"].apply(normalize_term)


# inner merge оставляет только те строки,
# где совпали и arXiv ID статьи, и сам термин.
intersection_df = pd.merge(
    tfidf_for_merge,
    keybert_for_merge,
    on=["arxiv_id", "normalized_term"],
    how="inner",
    suffixes=("_tfidf", "_keybert"),
)


intersection_df.to_csv(
    DATA_DIR / "tfidf_keybert_intersection.csv",
    index=False,
    encoding="utf-8-sig",
)


# Считаем итоговые показатели.
n_tfidf = tfidf_for_merge["normalized_term"].nunique()
n_keybert = keybert_for_merge["normalized_term"].nunique()
n_intersection = intersection_df["normalized_term"].nunique()
n_articles_overlap = intersection_df["arxiv_id"].nunique()


print("Уникальных TF-IDF терминов:", n_tfidf)
print("Уникальных KeyBERT терминов:", n_keybert)
print("Уникальных совпавших терминов:", n_intersection)
print(
    "Июльских статей хотя бы с одним совпадением:",
    n_articles_overlap,
    "из 50",
)


if len(intersection_df) > 0:
    display(intersection_df.head(30))


Уникальных TF-IDF терминов: 2
Уникальных KeyBERT терминов: 14990
Уникальных совпавших терминов: 0
Июльских статей хотя бы с одним совпадением: 0 из 50


### 8.1. Проверка влияния порога TF-IDF

При первоначальном анализе порог TF-IDF > 0.6 оказался очень строгим: было найдено только два уникальных термина.

В основном анализе используется исходный порог TF-IDF > 0.6. Поскольку при нём было найдено только два уникальных термина, дополнительно проводится анализ чувствительности результата к значениям 0.5, 0.4, 0.3, 0.2 и 0.1

Для каждого порога сравним:
- количество уникальных TF-IDF терминов;
- количество терминов, совпавших с результатами KeyBERT;
- количество июльских статей хотя бы с одним совпадением.

Корпус, параметры TfidfVectorizer и результаты KeyBERT при этом не изменяются.

In [14]:
# Пороги TF-IDF, которые хотим проверить
thresholds = [0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

# Здесь сохраним TF-IDF значения для всех июльских статей
all_tfidf_rows = []


# Сначала считаем TF-IDF только один раз для каждой статьи
for article_number, (_, article) in enumerate(july_df.iterrows(), start=1):

    # 50 предыдущих статей + одна июльская статья
    documents = previous_texts.copy()
    documents.append(article["analysis_text"])

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.95,
    )

    tfidf_matrix = vectorizer.fit_transform(documents)

    terms = vectorizer.get_feature_names_out()

    # Последняя строка относится к текущей июльской статье
    july_scores = tfidf_matrix[-1].toarray()[0]


    # Нам интересны значения выше самого низкого
    # проверяемого порога - 0.1
    for term_index in range(len(terms)):

        score = july_scores[term_index]

        if score > 0.1:

            term = terms[term_index]

            all_tfidf_rows.append({
                "arxiv_id": article["arxiv_id"],
                "term": term,
                "tfidf_score": float(score),
            })


    print(f"TF-IDF рассчитан для статьи {article_number}/50")


# Создаём одну общую таблицу со всеми найденными значениями
all_tfidf_scores = pd.DataFrame(all_tfidf_rows)

all_tfidf_scores["normalized_term"] = (
    all_tfidf_scores["term"].apply(normalize_term)
)


print()
print("TF-IDF рассчитан для всех 50 статей.")
print("Теперь проверяем разные пороги...")


# Теперь TF-IDF больше не пересчитываем.
# Просто фильтруем уже готовую таблицу.
threshold_results = []


for threshold in thresholds:

    # Оставляем значения выше текущего порога
    tfidf_test = all_tfidf_scores[
        all_tfidf_scores["tfidf_score"] > threshold
    ].copy()


    # Сравниваем их с результатами KeyBERT
    overlap_test = pd.merge(
        tfidf_test,
        keybert_for_merge,
        on=["arxiv_id", "normalized_term"],
        how="inner",
    )


    # Считаем результаты
    number_tfidf = tfidf_test["normalized_term"].nunique()

    number_overlap = overlap_test["normalized_term"].nunique()

    articles_overlap = overlap_test["arxiv_id"].nunique()


    threshold_results.append({
        "TF-IDF threshold": threshold,
        "TF-IDF terms": number_tfidf,
        "Intersection terms": number_overlap,
        "Articles with intersection": articles_overlap,
    })


    print(
        f"Порог {threshold}: "
        f"TF-IDF терминов = {number_tfidf}, "
        f"совпадений = {number_overlap}, "
        f"статей с совпадением = {articles_overlap}"
    )


# Итоговая таблица
threshold_table = pd.DataFrame(threshold_results)

print()
threshold_table.columns = [
    "Порог TF-IDF",
    "TF-IDF термины",
    "Совпавшие термины",
    "Статьи с совпадением",
]

display(threshold_table)

TF-IDF рассчитан для статьи 1/50
TF-IDF рассчитан для статьи 2/50
TF-IDF рассчитан для статьи 3/50
TF-IDF рассчитан для статьи 4/50
TF-IDF рассчитан для статьи 5/50
TF-IDF рассчитан для статьи 6/50
TF-IDF рассчитан для статьи 7/50
TF-IDF рассчитан для статьи 8/50
TF-IDF рассчитан для статьи 9/50
TF-IDF рассчитан для статьи 10/50
TF-IDF рассчитан для статьи 11/50
TF-IDF рассчитан для статьи 12/50
TF-IDF рассчитан для статьи 13/50
TF-IDF рассчитан для статьи 14/50
TF-IDF рассчитан для статьи 15/50
TF-IDF рассчитан для статьи 16/50
TF-IDF рассчитан для статьи 17/50
TF-IDF рассчитан для статьи 18/50
TF-IDF рассчитан для статьи 19/50
TF-IDF рассчитан для статьи 20/50
TF-IDF рассчитан для статьи 21/50
TF-IDF рассчитан для статьи 22/50
TF-IDF рассчитан для статьи 23/50
TF-IDF рассчитан для статьи 24/50
TF-IDF рассчитан для статьи 25/50
TF-IDF рассчитан для статьи 26/50
TF-IDF рассчитан для статьи 27/50
TF-IDF рассчитан для статьи 28/50
TF-IDF рассчитан для статьи 29/50
TF-IDF рассчитан для ст

,Порог TF-IDF,TF-IDF термины,Совпавшие термины,Статьи с совпадением
0,0.6,2,0,0
1,0.5,6,1,1
2,0.4,14,1,1
3,0.3,47,3,3
4,0.2,120,8,6
5,0.1,611,52,25


## 9. Итоговые результаты

При исходных параметрах анализа порог TF-IDF > 0.6 оказался достаточно строгим: было найдено 2 уникальных TF-IDF термина, и точных совпадений с результатами KeyBERT при пороге > 0.4 не обнаружено.

Дополнительная проверка показала, что количество найденных TF-IDF терминов и их пересечение с KeyBERT заметно зависит от выбранного порога. При снижении порога TF-IDF количество совпадений постепенно увеличивается.

Таким образом, отсутствие пересечения при исходном пороге 0.6 связано в том числе со строгостью отбора TF-IDF терминов. Полученные результаты показывают важность выбора порогового значения при сопоставлении результатов двух методов.

In [15]:
# Основные результаты при исходных порогах
main_results = {
    "Публикации за июль": 50,
    "Публикации за предыдущие два года": 50,
    "TF-IDF threshold": 0.6,
    "KeyBERT threshold": 0.4,
    "Уникальные TF-IDF термины": n_tfidf,
    "Уникальные KeyBERT термины": n_keybert,
    "Уникальные термины в пересечении": n_intersection,
    "Статьи хотя бы с одним совпадением": n_articles_overlap,
}

main_results_table = pd.DataFrame(
    list(main_results.items()),
    columns=["Показатель", "Значение"],
)

print("Основной анализ:")
display(main_results_table)

print()
print("Проверка разных порогов TF-IDF:")
display(threshold_table)

Основной анализ:


,Показатель,Значение
0,Публикации за июль,50.0
1,Публикации за предыдущие два года,50.0
2,TF-IDF threshold,0.6
3,KeyBERT threshold,0.4
4,Уникальные TF-IDF термины,2.0
5,Уникальные KeyBERT термины,14990.0
6,Уникальные термины в пересечении,0.0
7,Статьи хотя бы с одним совпадением,0.0



Проверка разных порогов TF-IDF:


,Порог TF-IDF,TF-IDF термины,Совпавшие термины,Статьи с совпадением
0,0.6,2,0,0
1,0.5,6,1,1
2,0.4,14,1,1
3,0.3,47,3,3
4,0.2,120,8,6
5,0.1,611,52,25
